# microGPT, on Wikipedia

Recreate Karpathy's microGPT, but trained on real Wikipedia text (`wikimedia/wikipedia`, Simple English) with our own byte-pair-encoding tokenizer and a GPT-2-style Transformer built from raw PyTorch tensor ops (`torch.autograd` for backward).

See `wiki/llm-from-scratch/microgpt.md` and `wiki/llm-from-scratch/nn-zero-to-hero.md`.

## [A] Setup & dependencies

Install deps with `uv` (`torch`, `datasets`) and do **all** of the notebook's imports here, then pick a device (CUDA / MPS / CPU) so the rest of the notebook stays device-agnostic.

In [ ]:
import math
import random
import time
from array import array
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import regex as re
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download

# Device-agnostic: prefer CUDA, then Apple MPS, else CPU.
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# One seed for reproducible runs across data sampling, init, and batching.
torch.manual_seed(1337)

print(f"torch {torch.__version__} | device: {device}")

## [B] Config

All hyperparameters in one place: data subset size, BPE vocab size, `block_size`, `n_layer` / `n_head` / `n_embd`, learning rate, and step count.

In [ ]:
# --- Data ---
n_articles = 180_000     # articles sampled into the corpus (~1 epoch of tokens for max_steps)
data_seed = 1337         # seed for the uniform random sample (reproducible corpus)

# --- Tokenizer (BPE) ---
vocab_size = 1024        # final vocab: 256 raw bytes + (vocab_size - 256) learned merges

# --- Model ---
block_size = 128         # context length (tokens the model attends over)
n_layer = 8              # number of Transformer blocks
n_head = 8               # attention heads per block
n_embd = 320             # embedding / residual-stream width (must be divisible by n_head)
assert n_embd % n_head == 0, "n_embd must be divisible by n_head"

# --- Training ---
batch_size = 32          # sequences per step
learning_rate = 3e-4     # peak Adam learning rate
max_steps = 20000        # total optimizer steps (x batch x block = tokens seen ~ 1 epoch)
warmup_steps = max(1, max_steps // 20)   # linear lr warmup, ~5% of training
min_lr = learning_rate * 0.1             # cosine-decay floor, 10% of peak
grad_clip = 1.0          # global L2 grad-norm ceiling (stability insurance)
eval_interval = 200      # steps between val-loss checks
eval_iters = 50          # batches averaged per val-loss estimate

print(f"model: {n_layer}L / {n_head}H / {n_embd}D | block {block_size} | vocab {vocab_size}")

## [C] Load Wikipedia

Download the full `20231101.simple` config to `data/` on first run (parquet via `snapshot_download`, which sidesteps the `datasets` resolution that stalls on this multi-config repo), then load it from local disk — re-runs do no network I/O. With the whole dataset in hand, take a seeded **uniform random sample** of N articles; sampling strategy can be iterated freely since the download + load stay cached. See `01_microgpt-wiki.notes.md`.

In [ ]:
# Anchor a gitignored data/ dir at the repo root (walk up to find pyproject.toml).
_here = Path.cwd().resolve()
repo_root = next(p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists())
data_dir = repo_root / "data"
data_dir.mkdir(exist_ok=True)

# First run: download the full Simple-English config's parquet locally. snapshot_download
# (targeted by allow_patterns) avoids the datasets resolution that stalls on this huge
# multi-config repo. Once the files are on disk, re-runs do no network I/O.
local_dir = data_dir / "wikipedia_simple"
config_dir = local_dir / "20231101.simple"
if not config_dir.exists() or not any(config_dir.glob("*.parquet")):
    print("downloading full 20231101.simple config (first run only) ...")
    snapshot_download(
        repo_id="wikimedia/wikipedia",
        repo_type="dataset",
        allow_patterns="20231101.simple/*.parquet",
        local_dir=str(local_dir),
    )

# Load the local parquet (memory-mapped, random-access) -- fast and offline after step 1.
parquet_files = sorted(str(p) for p in config_dir.glob("*.parquet"))
ds = load_dataset("parquet", data_files=parquet_files, split="train")
print(f"dataset: {len(ds):,} articles from {len(parquet_files)} local parquet file(s)")

# Seeded uniform random sample over the WHOLE dataset; iterate freely by changing
# data_seed / n_articles -- the download + load above stay cached.
idx = random.Random(data_seed).sample(range(len(ds)), n_articles)
articles = [ds[i]["text"] for i in idx]
corpus = "\n\n".join(articles)
print(f"corpus: {n_articles:,} of {len(ds):,} articles | {len(corpus):,} chars")

## [D] Inspect the corpus

Sanity-check the data: total size, number of articles, and a sample article.

In [ ]:
used = articles[:n_articles]
lengths = [len(a) for a in used]

print(f"articles:        {len(used):,}")
print(f"total chars:     {len(corpus):,}  (~{len(corpus) / 1e6:.1f}M)")
print(f"total bytes:     {len(corpus.encode('utf-8')):,}")
print(f"chars/article:   min {min(lengths):,} | median {sorted(lengths)[len(lengths) // 2]:,} | max {max(lengths):,}")
print(f"distinct chars:  {len(set(corpus)):,}")

# Show one representative article (the median-length one), trimmed.
median_idx = sorted(range(len(used)), key=lambda i: lengths[i])[len(used) // 2]
sample = used[median_idx]
print(f"\n--- sample article (#{median_idx}, {len(sample):,} chars) ---")
print(sample[:800] + ("..." if len(sample) > 800 else ""))

## [E] Pre-tokenization pattern

Before BPE, split text on a GPT-2-style `regex` so merges only ever combine a pair *within* a word / number / punctuation chunk, never across a space. This kills cross-word artifacts (the `", American"` story in the notes) **and** lets [H] cache encoding per unique chunk. The pattern uses `\p{L}` / `\p{N}` unicode classes whose alternation is **gap-free** — `pretokenize` partitions the text with no dropped characters, the correctness property the byte-level attempt lacked.

In [ ]:
# GPT-2's pre-tokenization regex (via the `regex` module, for its \p{L}/\p{N} unicode
# classes). The alternation covers every character -- contractions, letters, numbers,
# whitespace, and the "everything else" complement -- so finditer partitions text GAP-FREE.
GPT2_PAT = re.compile(
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)


def pretokenize(text):
    """text -> list of chunk bytestrings; concatenated they reconstruct text exactly."""
    return [m.group().encode("utf-8") for m in GPT2_PAT.finditer(text)]


# Low-level BPE primitives, shared by the [F] trainer and [G] encoder.
def get_stats(ids):
    """Count adjacent token pairs within one chunk."""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """Replace every occurrence of `pair` with a single new token `idx` (one L->R pass)."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


# Demo on real data: the first 1000 chars of the first article, then the pre-token chunks it
# splits into. (BPE token ids come later in [G]; here we only see the regex pre-tokens.)
_demo = articles[0][:1000]
print(_demo)
_chunks = pretokenize(_demo)
assert b"".join(_chunks) == _demo.encode("utf-8")   # gap-free: chunks rebuild the text exactly
print(f"\n--- {len(_chunks)} pre-token chunks (gap-free) ---")
print([c.decode("utf-8", "replace") for c in _chunks])

# Longest pre-tokens across the WHOLE corpus. A \p{L}+ / \p{N}+ run has no internal boundary,
# so run-together names, long digit strings, and camelCase markup each become one long chunk.
# Pure-whitespace runs are the literal longest but uninformative, so drop them. (~15s at 180k.)
_uniq = {c for c in (m.group() for m in GPT2_PAT.finditer(corpus)) if c.strip()}
_by_len = defaultdict(list)
for c in _uniq:
    _by_len[len(c)].append(c)
print(f"\n--- longest non-whitespace pre-tokens ({len(_uniq):,} unique) ---")
for L in sorted(_by_len, reverse=True)[:8]:
    print(f"len {L:3d} ({len(_by_len[L])}): " + "  ".join(repr(c) for c in sorted(_by_len[L])))


## [F] Train a minimal BPE

Learn `vocab_size - 256` merges over the **whole corpus**, counting adjacent pairs across pre-token chunks and only ever merging *within* a chunk. The corpus is first deduped into unique chunks with frequencies (they grow sublinearly — ~0.9M uniques for 180k articles), and pair counts are maintained **incrementally** — after each merge only the chunks that contained the merged pair are re-scanned (minbpe-style). Together that makes full-corpus training ~seconds (~35s at 180k), so the vocab's frequency tail stays representative rather than estimated from a small slice. Then the same demo as [E], now at the BPE level: the first article tokenized by the trained merges, and the longest learned tokens by length.

In [ ]:
# Incremental within-chunk BPE over the WHOLE corpus. Dedup into unique chunks with
# frequencies first -- unique chunks grow sublinearly (~0.9M for 180k articles), so the merge
# loop stays cheap -- then repeatedly merge the most frequent adjacent pair, re-touching only
# the chunks that contained it. Training on the full corpus (not a slice) keeps the vocab's
# frequency tail representative; dedup + incremental counts make it ~seconds, not hours.
num_merges = vocab_size - 256

word_freqs = Counter(pretokenize(corpus))       # unique chunk -> global frequency
words = [list(w) for w in word_freqs]           # each unique chunk as a list of byte ids
freqs = [word_freqs[w] for w in word_freqs]

# pair_counts: freq-weighted count of each adjacent pair; where[pair] = chunk indices with it.
pair_counts, where = {}, {}
for i, w in enumerate(words):
    for p in zip(w, w[1:]):
        pair_counts[p] = pair_counts.get(p, 0) + freqs[i]
        where.setdefault(p, set()).add(i)

merges = {}  # (int, int) -> int, insertion order == merge rank
t0 = time.time()
for k in range(num_merges):
    if not pair_counts:
        break
    pair = max(pair_counts, key=pair_counts.get)   # most frequent adjacent pair
    idx = 256 + k
    merges[pair] = idx
    for i in list(where.get(pair, ())):            # only chunks that contain `pair`
        w, f = words[i], freqs[i]
        old = Counter(zip(w, w[1:]))
        nw = merge(w, pair, idx)
        new = Counter(zip(nw, nw[1:]))
        for p in set(old) | set(new):              # update counts + index for changed pairs
            d = new[p] - old[p]
            if d:
                pair_counts[p] = pair_counts.get(p, 0) + f * d
                if pair_counts[p] <= 0:
                    pair_counts.pop(p, None)
                    where.get(p, set()).discard(i)
            if p in new and p not in old:
                where.setdefault(p, set()).add(i)
            elif p in old and p not in new:
                where.get(p, set()).discard(i)
        words[i] = nw
    pair_counts.pop(pair, None)
    where.pop(pair, None)

print(f"learned {len(merges)} merges from {len(word_freqs):,} unique chunks "
      f"({n_articles:,} articles) in {time.time() - t0:.1f}s")
print(f"final vocab size: {256 + len(merges)}")


# Build the id -> bytes table (the trainer's other output); [G] reuses it for the codec.
vocab = {i: bytes([i]) for i in range(256)}
for (p0, p1), i in merges.items():        # insertion-ordered: children built after parents
    vocab[i] = vocab[p0] + vocab[p1]

# Same demo as [E], now at the BPE level: the first 1000 chars of the first article, tokenized.
# (Inlines the greedy merge just to illustrate [F]; [G] defines the cached, reusable encode().)
_demo = articles[0][:1000]
_ids = []
for _chunk in pretokenize(_demo):
    _piece = list(_chunk)
    while len(_piece) >= 2:
        _pair = min(get_stats(_piece), key=lambda p: merges.get(p, float("inf")))
        if _pair not in merges:
            break
        _piece = merge(_piece, _pair, merges[_pair])
    _ids.extend(_piece)
print(_demo)
print(f"\n--- {len(_demo.encode('utf-8'))} bytes -> {len(_ids)} BPE tokens ---")
print("_".join(vocab[i].decode("utf-8", "replace") for i in _ids))

# Longest learned tokens by length -- the merged vocabulary [F] just produced.
_by_len = defaultdict(list)
for i, b in vocab.items():
    if len(b) > 1:
        _by_len[len(b)].append(b.decode("utf-8", "replace"))
print("\n--- longest learned tokens by length ---")
for L in sorted(_by_len, reverse=True)[:8]:
    print(f"len {L:2d} ({len(_by_len[L])}): " + "  ".join(repr(t) for t in sorted(_by_len[L])))


## [G] Encode / decode

`encode` pre-tokenizes, then greedily applies merges by rank *within* each chunk; results are **cached per unique chunk** (`_chunk_cache`), so [H] pays each unique pre-token's merge cost only once. `decode` inverts via the id→bytes `vocab` table built in [F]. Verify a roundtrip on sample text.

In [ ]:
# The codec, against the `vocab` / `merges` produced by [F].
def decode(ids):
    """ids -> text: concatenate each token's bytes, then UTF-8 decode."""
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")


_chunk_cache = {}   # chunk bytes -> token ids; populated lazily, reused across [G]/[H]


def encode_chunk(chunk):
    """One pre-token chunk (bytes) -> ids: greedily merge by learned rank. Cached."""
    cached = _chunk_cache.get(chunk)
    if cached is not None:
        return cached
    ids = list(chunk)
    while len(ids) >= 2:
        # pick the pair whose merge was learned earliest (lowest rank)
        pair = min(get_stats(ids), key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break                          # nothing left we know how to merge
        ids = merge(ids, pair, merges[pair])
    _chunk_cache[chunk] = ids
    return ids


def encode(text):
    """text -> ids: pre-tokenize, then encode each chunk (merges never cross chunks)."""
    out = []
    for m in GPT2_PAT.finditer(text):
        out.extend(encode_chunk(m.group().encode("utf-8")))
    return out


sample = "The quick brown fox jumps over the lazy dog. \U0001f98a"
demo_ids = encode(sample)
print(f"text:      {sample!r}")
print(f"lengths:   {len(sample.encode('utf-8'))} bytes -> {len(demo_ids)} tokens")
print(f"roundtrip: {decode(demo_ids) == sample}\n")

# Show the decoded text with token boundaries marked by underscores.
pieces = [vocab[i].decode("utf-8", errors="replace") for i in demo_ids]
print("_".join(pieces))

## [H] Tokenize the corpus

Stream pre-tokens across the whole corpus through the [G] chunk cache into a compact `uint16` buffer (each unique chunk encoded once), then wrap it as a tensor. Assert a **full-corpus roundtrip** — `decode(tokenize(corpus)) == corpus` — the correctness gate that guards against silently dropping characters. Split 90/10 into train / val.

In [ ]:
# Stream pre-tokens over the full corpus (finditer is lazy -- no giant intermediate list),
# encoding each unique chunk once via the [G] cache, into a compact uint16 buffer. vocab_size
# (1024) fits in uint16, so the ~90M-token tensor is ~2 bytes/token instead of 8.
t0 = time.time()
buf = array("H")
for m in GPT2_PAT.finditer(corpus):
    buf.extend(encode_chunk(m.group().encode("utf-8")))
data = torch.frombuffer(buf, dtype=torch.uint16).clone()   # clone so the tensor owns its memory
print(f"tokenized corpus: {len(data):,} tokens in {time.time() - t0:.1f}s "
      f"| {len(corpus.encode('utf-8')) / len(data):.2f} bytes/token "
      f"| {len(_chunk_cache):,} unique chunks cached")

# Correctness gate: the tokens must decode back to the EXACT corpus (no dropped chars).
assert decode(buf) == corpus, "roundtrip failed -- tokenization is lossy!"
print("full-corpus roundtrip: True")

# 90/10 train/val split on the contiguous token stream (val is the tail).
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"train: {len(train_data):,} tokens | val: {len(val_data):,} tokens")

## [I] Batching

`get_batch`: sample random contiguous (x, y) chunks of length `block_size` for next-token prediction.

In [ ]:
# Sample a batch of next-token training pairs. Pick batch_size random start offsets, take a
# length-block_size chunk x and the same chunk shifted by one as the targets y. The full data
# tensor stays on CPU as uint16 (compact); cast just the batch to long (embedding/index dtype)
# and move only it to `device`.
def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + 1 + block_size] for i in ix])
    return x.to(device).long(), y.to(device).long()


xb, yb = get_batch("train")
print(f"x: {tuple(xb.shape)} {xb.dtype} on {xb.device} | y: {tuple(yb.shape)}")
# y is x shifted by one: each position predicts the next token.
print(f"x[0, :8]: {xb[0, :8].tolist()}")
print(f"y[0, :8]: {yb[0, :8].tolist()}")

## [J] Embeddings

Token + position embedding tables built from raw tensors.

In [ ]:
# Two learned lookup tables, built from raw tensors. The token embedding maps each of the
# vocab_size ids to an n_embd vector; the position embedding gives each of the block_size
# slots its own n_embd vector. A token's input representation is the sum of the two.
# Small-normal init (std 0.02, GPT-2 style); requires_grad so autograd trains them.
def normal(*shape, std=0.02):
    return torch.randn(*shape, device=device) * std


tok_emb = normal(vocab_size, n_embd).requires_grad_()   # (vocab_size, n_embd)
pos_emb = normal(block_size, n_embd).requires_grad_()    # (block_size, n_embd)

# Embed the batch from [I]: look up token vectors, add the position vectors (broadcast over
# the batch), giving the (B, T, n_embd) tensor that enters the Transformer stack.
B, T = xb.shape
x = tok_emb[xb] + pos_emb[:T]
print(f"tok_emb: {tuple(tok_emb.shape)} | pos_emb: {tuple(pos_emb.shape)}")
print(f"embedded batch: {tuple(x.shape)} on {x.device}")

## [K] RMSNorm

Root-mean-square normalization for training stability, from raw tensor ops.

In [ ]:
# RMSNorm: rescale each token's n_embd vector to unit root-mean-square, then apply a learned
# per-channel gain. Unlike LayerNorm it does NOT subtract the mean or add a bias -- just the
# RMS rescale -- which is cheaper and what modern GPTs (LLaMA-style) use. Operates on the last
# dim independently per (batch, position), so it never mixes tokens.
def rmsnorm(x, weight, eps=1e-5):
    rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + eps)
    return x / rms * weight


# One gain vector per norm site; init to ones so the layer starts as a pure normalizer.
def norm_weight():
    return torch.ones(n_embd, device=device).requires_grad_()


# Sanity check on the [J] embeddings: post-norm rows should have ~unit RMS.
w = norm_weight()
xn = rmsnorm(x, w)
rms_after = xn.pow(2).mean(-1).sqrt()
print(f"in:  {tuple(x.shape)} | RMS mean {x.pow(2).mean(-1).sqrt().mean():.3f}")
print(f"out: {tuple(xn.shape)} | RMS mean {rms_after.mean():.3f} (~1.0)")

## [L] Multi-head causal self-attention

The heart of the Transformer: Q/K/V projections, scaled dot-product attention with a causal mask, and output projection — all raw tensor math.

In [ ]:
# Causal self-attention, raw tensor math. One c_attn matrix projects each token to its
# query/key/value (3*n_embd wide, split apart), and c_proj mixes the heads' outputs back to
# n_embd. head_dim = n_embd // n_head; the heads run in parallel as a tensor dimension.
def attn_weights():
    return {
        "c_attn": normal(n_embd, 3 * n_embd).requires_grad_(),  # q,k,v in one matmul
        "c_proj": normal(n_embd, n_embd).requires_grad_(),      # output projection
    }


def attention(x, w):
    B, T, C = x.shape
    head_dim = C // n_head
    q, k, v = (x @ w["c_attn"]).split(n_embd, dim=2)            # each (B, T, C)
    # (B, T, C) -> (B, n_head, T, head_dim): pull heads out as their own axis.
    q = q.view(B, T, n_head, head_dim).transpose(1, 2)
    k = k.view(B, T, n_head, head_dim).transpose(1, 2)
    v = v.view(B, T, n_head, head_dim).transpose(1, 2)
    # Scaled dot-product scores, then a causal mask so position t sees only <= t.
    att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)       # (B, n_head, T, T)
    causal = torch.tril(torch.ones(T, T, device=x.device))
    att = att.masked_fill(causal == 0, float("-inf"))
    att = torch.softmax(att, dim=-1)
    y = att @ v                                                 # (B, n_head, T, head_dim)
    y = y.transpose(1, 2).reshape(B, T, C)                      # reassemble heads
    return y @ w["c_proj"]


aw = attn_weights()
y = attention(xn, aw)
print(f"in: {tuple(xn.shape)} -> attn out: {tuple(y.shape)} on {y.device}")
# Causality check: position 0's attention row must put all weight on itself.
B, T, C = xn.shape
q0, k0, _ = (xn @ aw["c_attn"]).split(n_embd, dim=2)
hd = C // n_head
q0 = q0.view(B, T, n_head, hd).transpose(1, 2)
k0 = k0.view(B, T, n_head, hd).transpose(1, 2)
row0 = torch.softmax(
    ((q0 @ k0.transpose(-2, -1)) / math.sqrt(hd)).masked_fill(
        torch.tril(torch.ones(T, T, device=xn.device)) == 0, float("-inf")
    ),
    dim=-1,
)[0, 0, 0]
print(f"pos-0 attention row: self={row0[0]:.3f}, sum-of-rest={row0[1:].sum():.3e}")

## [M] MLP block

Per-position feed-forward block (up-projection, nonlinearity, down-projection).

In [ ]:
# Per-position feed-forward block: up-project n_embd -> 4*n_embd, apply a GELU nonlinearity,
# then down-project back to n_embd. Acts on each token independently (no mixing across
# positions -- that's attention's job); the 4x hidden is the standard GPT widening factor.
def mlp_weights(mult=4):
    return {
        "c_fc": normal(n_embd, mult * n_embd).requires_grad_(),    # up-projection
        "c_proj": normal(mult * n_embd, n_embd).requires_grad_(),  # down-projection
    }


def gelu(x):
    # Tanh approximation of GELU (the GPT-2 variant), from raw ops.
    return 0.5 * x * (1.0 + torch.tanh(
        math.sqrt(2.0 / math.pi) * (x + 0.044715 * x.pow(3))
    ))


def mlp(x, w):
    return gelu(x @ w["c_fc"]) @ w["c_proj"]


mw = mlp_weights()
h = mlp(xn, mw)
print(f"in: {tuple(xn.shape)} -> mlp out: {tuple(h.shape)} on {h.device}")
print(f"hidden width: {mw['c_fc'].shape[1]} (= {mw['c_fc'].shape[1] // n_embd}x n_embd)")

## [N] Transformer block

Compose one block: RMSNorm + attention + residual, then RMSNorm + MLP + residual.

In [ ]:
# One Transformer block, pre-norm style (norm BEFORE each sub-layer, GPT-2/LLaMA convention).
# Each sub-layer's output is ADDED back to its input -- the residual stream -- so x keeps its
# (B, T, n_embd) shape and gradients flow straight through. Bundle this block's four weight
# groups (two norm gains, attention, MLP) so [O] can stack n_layer of them.
def block_weights():
    return {
        "ln1": norm_weight(),
        "attn": attn_weights(),
        "ln2": norm_weight(),
        "mlp": mlp_weights(),
    }


def block(x, w):
    x = x + attention(rmsnorm(x, w["ln1"]), w["attn"])   # attention sub-layer + residual
    x = x + mlp(rmsnorm(x, w["ln2"]), w["mlp"])          # MLP sub-layer + residual
    return x


bw = block_weights()
xb_out = block(x, bw)   # feed the raw [J] embeddings through a full block
print(f"in: {tuple(x.shape)} -> block out: {tuple(xb_out.shape)} on {xb_out.device}")
# Residual sanity: output should be input plus a (smaller) update, not a wholesale rewrite.
delta = (xb_out - x).norm() / x.norm()
print(f"relative update ||block(x) - x|| / ||x|| = {delta:.3f}")

## [O] GPT model

Assemble embeddings + a stack of blocks + final norm + LM head into a forward pass returning logits and cross-entropy loss.

In [ ]:
# The whole model's parameters in one nested dict: embeddings, n_layer blocks, a final norm.
# The LM head is WEIGHT-TIED to tok_emb (logits = x @ tok_emb.T), so the same matrix embeds
# tokens in and projects back out -- standard for GPT-2/microGPT, and saves vocab_size*n_embd
# params. Residual-output projections are shrunk by 1/sqrt(2*n_layer) (GPT-2 init) so the
# residual stream's variance doesn't grow with depth.
def gpt_params():
    p = {
        "tok_emb": normal(vocab_size, n_embd).requires_grad_(),
        "pos_emb": normal(block_size, n_embd).requires_grad_(),
        "blocks": [block_weights() for _ in range(n_layer)],
        "ln_f": norm_weight(),
    }
    scale = 1.0 / math.sqrt(2 * n_layer)
    with torch.no_grad():
        for blk in p["blocks"]:
            blk["attn"]["c_proj"].mul_(scale)
            blk["mlp"]["c_proj"].mul_(scale)
    return p


def cross_entropy(logits, targets):
    # Raw, numerically-stable cross-entropy: log-softmax then gather the target log-probs.
    BT, V = logits.shape
    logp = logits - logits.logsumexp(-1, keepdim=True)
    return -logp[torch.arange(BT, device=logits.device), targets].mean()


def gpt_forward(params, idx, targets=None):
    B, T = idx.shape
    x = params["tok_emb"][idx] + params["pos_emb"][:T]    # embed (B, T, n_embd)
    for blk in params["blocks"]:
        x = block(x, blk)                                 # n_layer Transformer blocks
    x = rmsnorm(x, params["ln_f"])                        # final norm
    logits = x @ params["tok_emb"].T                      # tied LM head -> (B, T, vocab_size)
    if targets is None:
        return logits, None
    loss = cross_entropy(logits.reshape(B * T, -1), targets.reshape(B * T))
    return logits, loss


# Flatten the nested dict into a flat list of leaf tensors (what the optimizer will update).
def all_params(params):
    flat = [params["tok_emb"], params["pos_emb"], params["ln_f"]]
    for blk in params["blocks"]:
        flat += [blk["ln1"], blk["ln2"], blk["attn"]["c_attn"], blk["attn"]["c_proj"],
                 blk["mlp"]["c_fc"], blk["mlp"]["c_proj"]]
    return flat


params = gpt_params()
logits, loss = gpt_forward(params, xb, yb)
n_params = sum(t.numel() for t in all_params(params))
print(f"params: {n_params:,} ({n_params / 1e6:.2f}M)")
print(f"logits: {tuple(logits.shape)} | loss: {loss.item():.3f}")
# Sanity: an untrained model is ~uniform over vocab, so loss should be ~ln(vocab_size).
print(f"expected init loss ~ln(vocab_size) = {math.log(vocab_size):.3f}")

## [P] Optimizer & LR schedule

Adam optimizer with a decaying learning-rate schedule.

In [ ]:
# Adam, implemented from raw tensors. Per parameter we keep running first/second moment
# averages (m, v); each step uses bias-corrected moments for an adaptive, per-weight step.
# All updates run under no_grad -- they mutate the weights, they are not part of the graph.
class Adam:
    def __init__(self, params, lr, betas=(0.9, 0.95), eps=1e-8):
        self.params = list(params)
        self.lr, self.eps = lr, eps
        self.b1, self.b2 = betas
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    @torch.no_grad()
    def step(self):
        self.t += 1
        bc1, bc2 = 1 - self.b1**self.t, 1 - self.b2**self.t   # bias-correction terms
        for p, m, v in zip(self.params, self.m, self.v):
            g = p.grad
            m.mul_(self.b1).add_(g, alpha=1 - self.b1)         # m = b1*m + (1-b1)*g
            v.mul_(self.b2).addcmul_(g, g, value=1 - self.b2)  # v = b2*v + (1-b2)*g^2
            p.addcdiv_(m / bc1, (v / bc2).sqrt() + self.eps, value=-self.lr)

    def zero_grad(self):
        for p in self.params:
            p.grad = None


# Learning-rate schedule: linear warmup, then cosine decay from peak down to min_lr.
# (warmup_steps / min_lr live in [B] with the other hyperparameters.)
def get_lr(step):
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    return min_lr + 0.5 * (learning_rate - min_lr) * (1 + math.cos(math.pi * progress))


optimizer = Adam(all_params(params), lr=learning_rate)
probe = [0, warmup_steps, max_steps // 2, max_steps - 1]
print(f"optimizer: Adam over {len(optimizer.params)} tensors | warmup {warmup_steps} steps")
print("lr schedule: " + " | ".join(f"step {s}: {get_lr(s):.2e}" for s in probe))

## [Q] Training loop

Run training steps with periodic validation-loss evaluation.

In [ ]:
# Average loss over a handful of batches per split, with grad tracking off (eval only).
@torch.no_grad()
def estimate_loss():
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xe, ye = get_batch(split)
            _, l = gpt_forward(params, xe, ye)
            losses[k] = l.item()
        out[split] = losses.mean().item()
    return out


# Global-norm gradient clipping, from raw tensors (same spirit as the Adam above): if the
# L2 norm of ALL grads together exceeds max_norm, scale every grad down by the same factor.
# Caps the size of any single update -- cheap insurance against loss spikes from a bad batch.
@torch.no_grad()
def clip_grad_norm(ps, max_norm):
    total = torch.sqrt(sum(p.grad.pow(2).sum() for p in ps))
    if total > max_norm:
        for p in ps:
            p.grad.mul_(max_norm / (total + 1e-6))
    return total


# Minimal autoregressive sampler for in-training previews (encode prompt -> append tokens one
# at a time, each drawn from the temperature-scaled next-token distribution, cropping context
# to block_size). [S] formalizes the reusable version; inlined here since [Q] runs first.
preview_prompt = "This famous phrase is an English-language pangram: The quick brown fox "


@torch.no_grad()
def sample_text(prompt, max_new_tokens=60, temperature=0.8):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        ctx = torch.tensor([ids[-block_size:]], device=device)
        logits, _ = gpt_forward(params, ctx)
        probs = torch.softmax(logits[0, -1] / temperature, dim=-1)
        ids.append(torch.multinomial(probs, 1).item())
    return decode(ids)


# The training loop: each step sets the scheduled lr, samples a batch, runs forward to get the
# loss, backprops (autograd fills each leaf's .grad), clips the global grad norm, and Adam
# updates the weights. Every eval_interval steps we estimate train/val loss, record it for the
# [R] curves, and print a short sample so we can watch the text become more coherent.
history = {"step": [], "train": [], "val": []}
t0 = time.time()
for step in range(max_steps):
    optimizer.lr = get_lr(step)
    xt, yt = get_batch("train")
    _, loss = gpt_forward(params, xt, yt)
    optimizer.zero_grad()
    loss.backward()
    clip_grad_norm(optimizer.params, grad_clip)
    optimizer.step()

    if step % eval_interval == 0 or step == max_steps - 1:
        m = estimate_loss()
        history["step"].append(step)
        history["train"].append(m["train"])
        history["val"].append(m["val"])
        print(f"step {step:4d} | lr {optimizer.lr:.2e} | "
              f"train {m['train']:.3f} | val {m['val']:.3f} | {time.time() - t0:.0f}s")
        print(f"    {sample_text(preview_prompt)!r}")

print(f"done: {max_steps} steps in {time.time() - t0:.0f}s")

## [R] Loss curves

Plot train / val loss over steps to see learning progress.

In [ ]:
# Plot the recorded train/val cross-entropy against step. The dashed line is the untrained
# baseline ln(vocab_size) -- where loss starts before any learning -- so the gap below it is
# how much the model has learned. Train below val is normal; a widening gap = overfitting.
steps = history["step"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(steps, history["train"], "-o", ms=3, label="train")
ax.plot(steps, history["val"], "-o", ms=3, label="val")
ax.axhline(math.log(vocab_size), ls="--", c="gray", lw=1,
           label=f"uniform baseline ln(V) = {math.log(vocab_size):.2f}")
ax.set_xlabel("step")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Training / validation loss")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print(f"final: train {history['train'][-1]:.3f} | val {history['val'][-1]:.3f}")

## [S] Sampling

Autoregressive generation with the standard decoding knobs: temperature, top-k, top-p (nucleus — keep the smallest set of tokens whose cumulative probability reaches p), and a repetition penalty that dampens tokens already in the context.

In [ ]:
# Canonical autoregressive sampler (the reusable version of [Q]'s inline preview). Encode the
# prompt, then repeatedly: crop context to the last block_size tokens, forward for next-token
# logits, and shape the distribution before drawing one token. Knobs, applied in order:
#   repetition_penalty > 1  -- dampen logits of tokens already in the context (GPT-2/CTRL
#                              style: divide positive logits, multiply negative ones)
#   temperature             -- scale logits (lower = sharper/greedier, higher = more random)
#   top_k                   -- keep only the k most likely tokens
#   top_p                   -- nucleus: keep the smallest set of tokens whose cumulative
#                              probability reaches p (adaptive alternative to a fixed k)
@torch.no_grad()
def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=None, top_p=None,
             repetition_penalty=None):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        ctx = torch.tensor([ids[-block_size:]], device=device)
        logits, _ = gpt_forward(params, ctx)
        logits = logits[0, -1]
        if repetition_penalty is not None:
            seen = torch.tensor(sorted(set(ids[-block_size:])), device=device)
            logits[seen] = torch.where(logits[seen] > 0,
                                       logits[seen] / repetition_penalty,
                                       logits[seen] * repetition_penalty)
        logits = logits / temperature
        if top_k is not None:
            kth = torch.topk(logits, min(top_k, logits.size(-1))).values[-1]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        if top_p is not None:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            probs = torch.softmax(sorted_logits, dim=-1)
            # drop tokens once the cumulative mass BEFORE them reaches top_p
            # (the top token's "mass before" is 0, so at least one always survives)
            drop = probs.cumsum(-1) - probs >= top_p
            logits[sorted_idx[drop]] = float("-inf")
        probs = torch.softmax(logits, dim=-1)
        ids.append(torch.multinomial(probs, 1).item())
    return decode(ids)


# Smoke-test the knobs on a short prompt (run after training for coherent text).
for label, kwargs in [("temperature 0.5", dict(temperature=0.5)),
                      ("temperature 1.0", dict(temperature=1.0)),
                      ("top_p 0.9", dict(top_p=0.9)),
                      ("top_p 0.9 + rep 1.3", dict(top_p=0.9, repetition_penalty=1.3))]:
    print(f"--- {label} ---")
    print(generate("The ", max_new_tokens=80, **kwargs))
    print()

## [T] Generate

Produce Wikipedia-style text samples from the trained model, comparing the fixed top-k cutoff against nucleus sampling with a repetition penalty on the same prompts.

In [ ]:
# Longer Wikipedia-style samples, same prompts under two decoding configs. Left: the fixed
# top_k=50 cutoff from before. Right: nucleus (top_p=0.9) -- the candidate set adapts to the
# model's confidence -- plus a mild repetition penalty (1.3) to discourage loops. Different
# prompts probe whether the model picked up encyclopedic phrasing and topic-setting openings.
prompts = ["The history of", "In 1969, ", "Water is "]
for p in prompts:
    print(f"=== {p!r} | top_k=50 ===")
    print(generate(p, max_new_tokens=200, temperature=0.8, top_k=50))
    print(f"\n=== {p!r} | top_p=0.9, repetition_penalty=1.3 ===")
    print(generate(p, max_new_tokens=200, temperature=0.8, top_p=0.9, repetition_penalty=1.3))
    print()

## [U] Save / load

Persist the BPE merges and model `state_dict` to `artifacts/` (gitignored) so a trained run can be reloaded without retraining.

In [ ]:
# Persist the trained tokenizer + model to a gitignored artifacts/ dir so a run can be reloaded
# without retraining. `merges` defines the BPE; `params` is our raw-tensor "state_dict" (a
# nested dict of leaf tensors). torch.save (pickle) handles the tuple-keyed merges and the
# nested tensors; we also stash the config so a reload can rebuild the right-shaped model.
artifacts_dir = repo_root / "artifacts"
artifacts_dir.mkdir(exist_ok=True)
ckpt_path = artifacts_dir / "microgpt_wiki.pt"


def _map(p, fn):
    """Recurse the nested params dict/list, applying fn to each leaf tensor."""
    if isinstance(p, dict):
        return {k: _map(v, fn) for k, v in p.items()}
    if isinstance(p, list):
        return [_map(v, fn) for v in p]
    return fn(p)


config = {k: globals()[k] for k in ("vocab_size", "block_size", "n_layer", "n_head", "n_embd")}
# Detach + move to CPU so the checkpoint is device-agnostic and carries no autograd graph.
torch.save(
    {"merges": merges, "params": _map(params, lambda t: t.detach().cpu()), "config": config},
    ckpt_path,
)
print(f"saved {ckpt_path.stat().st_size / 1e6:.1f} MB -> {ckpt_path.relative_to(repo_root)}")

# Reload round-trip: rebuild leaves on the current device, then confirm an identical forward.
ckpt = torch.load(ckpt_path, weights_only=False)
loaded = _map(ckpt["params"], lambda t: t.to(device).requires_grad_())
xv, yv = get_batch("val")
_, l_live = gpt_forward(params, xv, yv)
_, l_loaded = gpt_forward(loaded, xv, yv)
print(f"reload check (same batch): live {l_live.item():.4f} vs loaded {l_loaded.item():.4f}")